# Starter Notebook — LLM Research Harness with Ollama on AWS

This notebook is a **starter scaffold** for the workshop. Its purpose is to help a pod move from:

- a research task
- a minimum viable context pack
- a minimum viable harness plan

to a **first working workflow** in Jupyter on HPC using Ollama.


## 1. Workshop task and pod notes

Fill these in before you start coding.

- **Task title:**
- **Task description:**
- **Input type:**
- **Desired output:**
- **One likely failure mode:**
- **Human review rule:**


## 2. Minimum viable context pack

Fill in or revise these for your pod’s chosen task.

- **Task statement:**
- **Sample input:**
- **Ideal output example:**
- **Schema / structure:**
- **One caution or guardrail:**


## 3. Context pack to LLM Harness

Fill in or revise these for your pod’s chosen task.

**The smallest repeatable set up around an LLM task**

**The harness is the code version of your context package**
- input
- prompt/template
- script/notebook (code)
- output file
- log/metadata


In [ ]:
from pathlib import Path
from datetime import datetime
import json

print('Python environment is ready.')


Python environment is ready.


In [ ]:
# For use on AWS use (ignore if you are using on HPC)

!sudo dnf install zstd  pciutils  lshw -y
!pip install torch numpy
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, requests

# Ollama is a server. Our instance has no init system, so background it ourselves
# and hold the handle for the life of the runtime.
server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Poll until it answers instead of sleeping a fixed amount.
for attempt in range(60):
    try:
        requests.get("http://127.0.0.1:11434/api/tags", timeout=1).raise_for_status()
        print(f"Ollama is up (took ~{attempt}s)")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start. Re-run this cell.")

import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    MODEL = "qwen3:8b" if vram_gb >= 12 else "qwen3:4b"
    print(f"GPU: {gpu} ({vram_gb:.1f} GB)")
else:
    MODEL = "qwen3:1.7b"
    print("No GPU assigned - falling back to CPU.")

print(f"Using model: {MODEL}")

In [ ]:
# If the following cell doesn't run, uncomment this line once to install the Python package:
!pip install -q ollama


In [ ]:


import ollama


OLLAMA_MODEL = 'qwen3:8b'  # insert you model of choice here later on
PROMPT_VERSION = 'v1'

print('OLLAMA_MODEL =', OLLAMA_MODEL)


OLLAMA_MODEL = qwen3:8b


## 4. Optional quick connectivity check

Run this only if you want to quickly check whether Ollama is reachable from the notebook.


In [ ]:
try:
    tags = ollama.list()
    print('Ollama reachable.')
    print(tags)
except Exception as e:
    print('Could not reach Ollama. If needed, start it from the terminal with `ollama serve &`.')

    print('Error:', e)


Could not reach Ollama. If needed, start it from the terminal with `ollama serve &`.
Error: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download


## 5. Example starter data

Use this small sample first if you want a quick test before swapping in your own data.

The default example is a simple **abstract extraction** task.


In [ ]:
sample_records = [
    {
        'id': 1,
        'text': 'This study examined the relationship between social support and academic persistence among first-generation college students at a large public university in the United States. Using survey data from 412 undergraduate students, the authors conducted multiple regression analyses to assess whether perceived support from family, peers, and faculty predicted students’ intention to remain enrolled. Results indicated that faculty support and peer support were significant positive predictors of persistence, while family support was not statistically significant. The findings suggest that university-based support systems may play an important role in retaining first-generation students.'
    },
    {
        'id': 2,
        'text': 'Researchers explored how graduate students used AI tools during the first semester of a doctoral program. Through semi-structured interviews with 18 students, the study identified themes related to experimentation, uncertainty, and uneven faculty guidance. The authors argue that AI literacy support should be embedded into early doctoral training.'
    },
    {
        'id': 3,
        'text': 'We tested whether a structured peer-mentoring intervention improved the weekly engagement of students in an online statistics course. Data from the learning management system were compared across intervention and control sections over ten weeks. Students in the intervention section showed higher discussion participation, but there was no significant difference in quiz scores.'
    }
]

print(f'Loaded {len(sample_records)} sample records.')
print(json.dumps(sample_records[0], indent=2)[:1200])


Loaded 3 sample records.
{
  "id": 1,
  "text": "This study examined the relationship between social support and academic persistence among first-generation college students at a large public university in the United States. Using survey data from 412 undergraduate students, the authors conducted multiple regression analyses to assess whether perceived support from family, peers, and faculty predicted students\u2019 intention to remain enrolled. Results indicated that faculty support and peer support were significant positive predictors of persistence, while family support was not statistically significant. The findings suggest that university-based support systems may play an important role in retaining first-generation students."
}


## 6. Define the prompt template

Start with this example and adapt it to your pod’s chosen task.

This version is for structured extraction from abstracts. You can revise the task statement, schema, and guardrail.


In [ ]:
task_statement = 'Extract key study information from the text into a structured record for a literature review table.'
guardrail = 'Use only information explicitly stated in the text. If something is missing, write "not stated" rather than guessing.'
output_schema = {
    'topic': 'string',
    'population': 'string',
    'method': 'string',
    'main_finding': 'string',
    'limitations_or_missing_info': 'string'
}

prompt_template = '''You are helping with a research workflow.

Task:
{task_statement}

Guardrail:
{guardrail}

Return the result in JSON with this structure:
{output_schema}

Text:
{input_text}
'''

print(prompt_template[:500])


You are helping with a research workflow.

Task:
{task_statement}

Guardrail:
{guardrail}

Return the result in JSON with this structure:
{output_schema}

Text:
{input_text}



## 7. Helper function: call Ollama

This is the core LLM step inside the workflow.

If you want, ask the chatbot to help explain or revise this function.


In [ ]:
def call_ollama(prompt: str, model: str = OLLAMA_MODEL) -> str:
    response = ollama.generate(model=model, prompt=prompt)
    return response['response']


## 8. Test on one example first

This is an important checkpoint. Before running over many inputs, test the workflow on one input and inspect the result.


In [ ]:
example_record = sample_records[0]
prompt = prompt_template.format(
    task_statement=task_statement,
    guardrail=guardrail,
    output_schema=json.dumps(output_schema, indent=2),
    input_text=example_record['text'],
)

print(prompt[:1200])


You are helping with a research workflow.

Task:
Extract key study information from the text into a structured record for a literature review table.

Guardrail:
Use only information explicitly stated in the text. If something is missing, write "not stated" rather than guessing.

Return the result in JSON with this structure:
{
  "topic": "string",
  "population": "string",
  "method": "string",
  "main_finding": "string",
  "limitations_or_missing_info": "string"
}

Text:
This study examined the relationship between social support and academic persistence among first-generation college students at a large public university in the United States. Using survey data from 412 undergraduate students, the authors conducted multiple regression analyses to assess whether perceived support from family, peers, and faculty predicted students’ intention to remain enrolled. Results indicated that faculty support and peer support were significant positive predictors of persistence, while family suppo

In [ ]:
# Uncomment to run once Ollama is ready.
test_output = call_ollama(prompt)
print(test_output)


Here is the extracted information in JSON format:

```json
{
  "topic": "The relationship between social support and academic persistence among first-generation college students",
  "population": "412 undergraduate students at a large public university in the United States",
  "method": "Multiple regression analyses using survey data",
  "main_finding": "Faculty and peer support are significant positive predictors of persistence, while family support is not statistically significant",
  "limitations_or_missing_info": "Not stated"
}
```

Note that I wrote "not stated" for the limitations or missing information section as there was no mention of any limitations or areas where additional research is needed in the provided text.


## 9. Optional: inspect or parse the output

If your model returns valid JSON, you can try to parse it. If not, keep the raw text for now.


In [ ]:
import json
import re

def safe_parse_json(text: str):
    if not text or not text.strip():
        raise ValueError("Model output is empty.")

    cleaned = text.strip()

    # First try direct parsing
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # Try fenced ```json ... ``` block
    fence_match = re.search(r"```json\s*(\{.*?\})\s*```", cleaned, re.DOTALL)
    if fence_match:
        return json.loads(fence_match.group(1))

    # Try any JSON object in the text
    object_match = re.search(r"(\{.*\})", cleaned, re.DOTALL)
    if object_match:
        return json.loads(object_match.group(1))

    raise ValueError("Could not find valid JSON in model output.")

In [ ]:
# Example pattern if the model returns valid JSON.
parsed = safe_parse_json(test_output)
print(json.dumps(parsed, indent=2))


{
  "topic": "The relationship between social support and academic persistence among first-generation college students",
  "population": "412 undergraduate students at a large public university in the United States",
  "method": "Multiple regression analyses using survey data",
  "main_finding": "Faculty and peer support are significant positive predictors of persistence, while family support is not statistically significant",
  "limitations_or_missing_info": "Not stated"
}


## 10. Run over multiple inputs

Once the single-example test looks reasonable, run the workflow over multiple records.

This scaffold saves both the raw model output and the original input id.


In [ ]:
results = []

for record in sample_records:
    prompt = prompt_template.format(
        task_statement=task_statement,
        guardrail=guardrail,
        output_schema=json.dumps(output_schema, indent=2),
        input_text=record['text'],
    )

    # Uncomment when ready to run against Ollama.
    # raw_output = call_ollama(prompt)

    # Temporary placeholder so the notebook structure works before the real call.
    raw_output = 'REPLACE_WITH_OLLAMA_OUTPUT'

    results.append({
        'id': record['id'],
        'input_text': record['text'],
        'raw_output': raw_output,
    })

print(f'Prepared {len(results)} result records.')
print(json.dumps(results[0], indent=2)[:1200])


Prepared 3 result records.
{
  "id": 1,
  "input_text": "This study examined the relationship between social support and academic persistence among first-generation college students at a large public university in the United States. Using survey data from 412 undergraduate students, the authors conducted multiple regression analyses to assess whether perceived support from family, peers, and faculty predicted students\u2019 intention to remain enrolled. Results indicated that faculty support and peer support were significant positive predictors of persistence, while family support was not statistically significant. The findings suggest that university-based support systems may play an important role in retaining first-generation students.",
  "raw_output": "REPLACE_WITH_OLLAMA_OUTPUT"
}


## 11. Save outputs

Saving the output file is part of the harness.


In [ ]:
output_dir = Path('workshop_outputs')
output_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_file = output_dir / f'results_{timestamp}.json'

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)

print('Saved output to:', output_file)


Saved output to: workshop_outputs/results_20260510_181047.json


## 12. Save metadata

Saving metadata makes the workflow easier to repeat, compare, and explain later.


In [ ]:
metadata = {
    'run_time': datetime.now().isoformat(),
    'model': OLLAMA_MODEL,
    'prompt_version': PROMPT_VERSION,
    'input_records': len(sample_records),
    'output_file': str(output_file),
    'task_statement': task_statement,
    'guardrail': guardrail,
}

metadata_file = output_dir / f'metadata_{timestamp}.json'
with open(metadata_file, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

print('Saved metadata to:', metadata_file)
print(json.dumps(metadata, indent=2))


Saved metadata to: workshop_outputs/metadata_20260510_181047.json
{
  "run_time": "2026-05-10T18:10:52.909354",
  "model": "llama3.1:8b",
  "prompt_version": "v1",
  "input_records": 3,
  "output_file": "workshop_outputs/results_20260510_181047.json",
  "task_statement": "Extract key study information from the text into a structured record for a literature review table.",
  "guardrail": "Use only information explicitly stated in the text. If something is missing, write \"not stated\" rather than guessing."
}


## 13. Optional: load your own JSON input

If you have your own data as JSON, adapt this section.

Suggested structure:
- a list of records
- each record has an `id` and a `text` field


In [ ]:
# Example pattern:
# with open('my_input.json', 'r', encoding='utf-8') as f:
#     my_records = json.load(f)
#
# print(f'Loaded {len(my_records)} records.')
# print(json.dumps(my_records[0], indent=2)[:1200])


## 14. Quick evaluation notes

Use this section to reflect before scaling.

- What looked useful?
- What failed?
- What would you revise in the prompt, schema, or workflow?
- What should a human review before using the outputs?


## 15. Optional next steps

If your pod finishes early, try one of these:

- replace the sample data with your own JSON input file
- revise the schema or guardrail
- parse the model output into cleaner structured JSON
- separate the prompt into its own `.txt` or `.md` file
- turn repeated notebook steps into Python functions
- add a simple validation check
- prepare a version that could later be run with `sbatch`
